# Chemical Space Explorer — Framework 2: Physicochemical Descriptors

## Mission
Identify physicochemically distinct property clusters across a compound library to guide
combinatorial building block selection. The goal is to select ~30 building blocks from
millions of enumerated structures such that their combinations maximally cover
physicochemical property space. K-Medoids medoids are the synthesis candidates.

This framework does **not** claim to represent 'chemical space' in the structural sense.
It represents **physicochemical property space** — the six descriptors capture size,
lipophilicity, polarity, and flexibility axes that define bRo5 (beyond Rule of Five)
drug-like property space. Structural diversity is addressed separately in Framework 1.

## Representation
Six physicochemical descriptors (RDKit): MolWt, MolLogP, TPSA, NumHAcceptors,
NumHDonors, NumRotatableBonds. These are the standard bRo5 descriptors used to
characterize macrocyclic and cyclic peptide drug-likeness in the literature.
No PCA applied — 6 features is below the threshold where PCA is beneficial, and
PCA would compress physically interpretable axes into abstract components, making
it impossible to ask 'which property drove this cluster' — the key scientific question.

## Pipeline
```
SMILES → 6 physicochemical descriptors
  → Tanimoto deduplication (exact + near-duplicate removal)
  → RobustScaler (median=0, IQR=1)
  → K-Medoids (cosine, k from elbow) ─┐ cluster in
  → HDBSCAN (euclidean)               ┘ descriptor space
  → UMAP (cosine, 1 seed) → 2D canvas (visualization only)
  → PCA 2D canvas (sanity check — if clusters look the same in PCA and UMAP,
    results are robust to visualization method choice)
  → Paint cluster labels onto both canvases
```

## QC Scores
- Silhouette (cosine) → K-Medoids quality
- DBCV (relative_validity_) → HDBSCAN quality
- PCA vs UMAP cluster consistency → visualization robustness
- Cosine inter-cluster distance → cluster separation

## Outputs
- `cluster_assignments_desc.csv` — per-compound cluster labels + UMAP XY
- `synthesis_candidates_desc.csv` — ranked medoids with coverage score
- 7 figures (including PCA sanity check)

---
**Note — future experiments (flag for follow-up):**
- HDBSCAN parameter sensitivity grid: omitted here due to scale. Run on subsampled
  dataset (~50K) first to tune min_cluster_size and min_samples.
- UMAP 5-seed stability check: single seed used here at 5M scale. Re-enable on
  subsampled data to validate layout reproducibility.
- Bootstrap cluster stability (80% subsampling × 20 runs): deferred to downstream
  validation after initial results reviewed.
- Mordred descriptor expansion: this framework uses 6 bRo5 descriptors as a first
  pass. Expanding to the full Mordred descriptor set (~1800 descriptors) with
  correlation filtering is the planned next iteration.

In [ ]:
!pip install -q umap-learn hdbscan scikit-learn-extra rdkit-pypi
print('Dependencies installed.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — only edit this cell
# ═══════════════════════════════════════════════════════════════════════════════

LIBRARY_CSV    = 'library.csv'
LITERATURE_CSV = 'literature.csv'   # None to skip

SMILES_COL     = 'smiles'
NAME_COL       = 'name'
ACTIVITY_COL   = 'ic50_nm'
ACTIVITY_LABEL = 'IC50 (nM)'
ACTIVITY_LOG   = True

# Physicochemical descriptors — bRo5 standard set
# These 6 descriptors cover: size (MolWt), lipophilicity (MolLogP),
# polarity (TPSA, HBA, HBD), and flexibility (RotBonds).
# No PCA applied: with ≤6 features, PCA would destroy the physical
# interpretability of individual descriptors.
DESCRIPTORS = [
    'MolWt',
    'MolLogP',
    'TPSA',
    'NumHAcceptors',
    'NumHDonors',
    'NumRotatableBonds',
]

# K-Medoids: set after running the elbow plot cell
N_KMEDOIDS     = 8

# HDBSCAN
HDBSCAN_MIN_SIZE  = 50
HDBSCAN_MIN_SAMP  = 10

# UMAP — cosine metric (matches K-Medoids metric, visualization only)
# Cosine distance chosen over Euclidean because it measures the angle between
# descriptor vectors, invariant to overall magnitude differences across
# compounds of different sizes — critical when clustering libraries spanning
# a wide MW range.
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST    = 0.1
RANDOM_STATE     = 42
# Note: single seed used here due to scale. See future experiments note.

# Tanimoto deduplication
DEDUP_TANIMOTO   = 0.95

# Inter-cluster diversity sample size
DIVERSITY_SAMPLE = 1000

MAX_COMPOUNDS    = None
OUTPUT_DIR       = 'results_desc'
# ═══════════════════════════════════════════════════════════════════════════════
print('Configuration loaded.')

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

import umap
import hdbscan as hdbscan_lib
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics.pairwise import cosine_distances
from sklearn_extra.cluster import KMedoids

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, Draw, DataStructs
from IPython.display import display

warnings.filterwarnings('ignore')
Path(OUTPUT_DIR).mkdir(exist_ok=True)
print('Imports OK.')

## 1. Load Data & Compute Descriptors

In [ ]:
_DESC_FN = {
    'MolWt':             Descriptors.MolWt,
    'MolLogP':           Descriptors.MolLogP,
    'TPSA':              Descriptors.TPSA,
    'NumHAcceptors':     Descriptors.NumHAcceptors,
    'NumHDonors':        Descriptors.NumHDonors,
    'NumRotatableBonds': Descriptors.NumRotatableBonds,
}

def load_dataset(path, label):
    df = pd.read_csv(path)
    df = df.rename(columns={SMILES_COL: 'smiles'})
    if NAME_COL and NAME_COL in df.columns:
        df = df.rename(columns={NAME_COL: 'name'})
    else:
        df['name'] = [f'{label}_{i}' for i in range(len(df))]
    if ACTIVITY_COL and ACTIVITY_COL in df.columns:
        df = df.rename(columns={ACTIVITY_COL: 'activity'})
    df['source'] = label
    rows = []
    parse_fail = []
    for i, smi in enumerate(df['smiles']):
        mol = Chem.MolFromSmiles(str(smi))
        if mol:
            rows.append({d: _DESC_FN[d](mol) for d in DESCRIPTORS})
        else:
            rows.append({d: np.nan for d in DESCRIPTORS})
            parse_fail.append(smi)
    desc_df = pd.DataFrame(rows)
    df = pd.concat([df.reset_index(drop=True), desc_df], axis=1)
    if parse_fail:
        print(f'  {label}: {len(parse_fail)} SMILES parse failures saved to {OUTPUT_DIR}/parse_failures.csv')
        pd.Series(parse_fail).to_csv(f'{OUTPUT_DIR}/parse_failures_{label}.csv', index=False)
    df = df.dropna(subset=DESCRIPTORS).reset_index(drop=True)
    print(f'  {label}: {len(df):,} compounds')
    return df

lib_df = load_dataset(LIBRARY_CSV, 'library')
if LITERATURE_CSV:
    lit_df = load_dataset(LITERATURE_CSV, 'literature')
    df = pd.concat([lib_df, lit_df], ignore_index=True)
else:
    df = lib_df.copy()

if MAX_COMPOUNDS and len(df) > MAX_COMPOUNDS:
    df = df.sample(MAX_COMPOUNDS, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'Subsampled to {MAX_COMPOUNDS:,} compounds.')

if 'activity' in df.columns:
    df['activity_plot'] = np.log10(df['activity'].clip(lower=1e-3)) if ACTIVITY_LOG else df['activity']

print(f'\nTotal: {len(df):,}  |  Sources: {df["source"].value_counts().to_dict()}')

## 2. Deduplication

Remove exact SMILES duplicates, then near-duplicates (Tanimoto > 0.95 on Morgan FP).
Prevents artificial inflation of cluster quality from highly similar compounds.

In [ ]:
n_before = len(df)
df = df.drop_duplicates(subset='smiles').reset_index(drop=True)
n_exact = n_before - len(df)
print(f'Exact duplicates removed: {n_exact:,}')

print('Computing Morgan fingerprints for near-duplicate detection...')
fps = []
for smi in df['smiles']:
    mol = Chem.MolFromSmiles(str(smi))
    fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048) if mol else None)
df['_fp'] = fps
df = df[df['_fp'].notna()].reset_index(drop=True)
fps = df['_fp'].tolist()

df = df.sort_values('source', ascending=False).reset_index(drop=True)
fps = df['_fp'].tolist()
keep = [True] * len(df)
kept_fps = []
BATCH_CHECK = 200
for i, fp in enumerate(fps):
    if not keep[i] or fp is None:
        continue
    if kept_fps:
        sims = DataStructs.BulkTanimotoSimilarity(fp, kept_fps[-BATCH_CHECK:])
        if max(sims) > DEDUP_TANIMOTO:
            keep[i] = False
            continue
    kept_fps.append(fp)

df = df[keep].reset_index(drop=True)
fps = df['_fp'].tolist()
n_neardup = sum(~np.array(keep))
print(f'Near-duplicates removed : {n_neardup:,}')
print(f'Final dataset           : {len(df):,} compounds')

## 3. Dataset Characterization

Descriptor distributions reveal what property space the library covers before clustering.
Bimodal distributions or extreme outliers will influence RobustScaler and clustering.
Activity distribution shows the potency range of the reference compounds.

In [ ]:
n_plots = len(DESCRIPTORS) + (1 if 'activity' in df.columns else 0)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, desc in enumerate(DESCRIPTORS):
    axes[i].hist(df[desc].dropna(), bins=40, color='steelblue', edgecolor='white')
    axes[i].set_title(desc)
    axes[i].set_xlabel('Value'); axes[i].set_ylabel('Count')
    med = df[desc].median()
    axes[i].axvline(med, color='red', linestyle='--', label=f'Median={med:.1f}')
    axes[i].legend(fontsize=8)

if 'activity_plot' in df.columns:
    axes[len(DESCRIPTORS)].hist(df['activity_plot'].dropna(), bins=40, color='coral', edgecolor='white')
    axes[len(DESCRIPTORS)].set_title(f'log10({ACTIVITY_LABEL})')
    axes[len(DESCRIPTORS)].set_xlabel('Value'); axes[len(DESCRIPTORS)].set_ylabel('Count')

for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Dataset Characterization — Descriptor Distributions', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dataset_characterization.png', dpi=150, bbox_inches='tight')
plt.show()

print('Descriptor summary:')
display(df[DESCRIPTORS].describe().round(2))

## 4. Normalization — RobustScaler

RobustScaler (median=0, IQR=1) was chosen over StandardScaler because MW and logP
distributions in drug-like libraries are heavy-tailed. Outlier compounds with extreme
values would inflate StandardScaler's mean and standard deviation, distorting cosine
distances for the majority of compounds. RobustScaler is resistant to these outliers
while still normalizing unit differences across heterogeneous descriptors.

In [ ]:
X = df[DESCRIPTORS].values
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
print(f'Normalized descriptor matrix: {X_scaled.shape}')
print('Post-scaling summary (should be centered near 0):')
display(pd.DataFrame(X_scaled, columns=DESCRIPTORS).describe().round(3))

## 5. Clustering in Descriptor Space

All clustering is performed on the normalized 6-descriptor matrix.
UMAP and PCA run later for visualization only.

Cosine distance for K-Medoids: invariant to overall descriptor magnitude, appropriate
for libraries spanning a wide MW range. HDBSCAN uses euclidean on the normalized space.

In [ ]:
# ── K-Medoids elbow: silhouette vs k ─────────────────────────────────────────
n_elbow = min(len(X_scaled), 10_000)
X_elbow = X_scaled[np.random.choice(len(X_scaled), n_elbow, replace=False)] if n_elbow < len(X_scaled) else X_scaled
print(f'Elbow computed on {n_elbow:,} sample.')

k_range = range(2, 21)
sil_scores = []
for k in k_range:
    km_t = KMedoids(n_clusters=k, metric='cosine', method='alternate', random_state=RANDOM_STATE)
    lbl  = km_t.fit_predict(X_elbow)
    sil_scores.append(silhouette_score(X_elbow, lbl, metric='cosine'))
    print(f'  k={k:2d}  silhouette={sil_scores[-1]:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_range), sil_scores, 'o-', color='steelblue', markersize=6)
ax.axvline(N_KMEDOIDS, color='red', linestyle='--', label=f'Current N_KMEDOIDS={N_KMEDOIDS}')
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score (cosine)')
ax.set_title('K-Medoids Elbow — Choose k at peak or first plateau')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/kmedoids_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Optimal k (max silhouette): {list(k_range)[np.argmax(sil_scores)]}')
print('Update N_KMEDOIDS in config if needed, then rerun next cell.')

In [ ]:
print(f'Running K-Medoids (k={N_KMEDOIDS}, cosine)...')
km = KMedoids(n_clusters=N_KMEDOIDS, metric='cosine', method='alternate', random_state=RANDOM_STATE)
km_labels  = km.fit_predict(X_scaled)
medoid_idx = list(km.medoid_indices_)
sil_km     = silhouette_score(X_scaled, km_labels, metric='cosine')
print(f'Silhouette (cosine): {sil_km:.4f}  ({"strong" if sil_km>0.5 else "reasonable" if sil_km>0.25 else "weak"})')
df['km_cluster'] = km_labels

In [ ]:
# DBCV is the correct metric for HDBSCAN — silhouette assumes convex clusters
# and gives misleadingly good scores for density-based clusters.
# Note: sensitivity grid omitted at scale — tune on 50K subsample first.
print('Running HDBSCAN...')
clusterer = hdbscan_lib.HDBSCAN(
    min_cluster_size       = HDBSCAN_MIN_SIZE,
    min_samples            = HDBSCAN_MIN_SAMP,
    metric                 = 'euclidean',
    cluster_selection_method = 'eom',
    gen_min_span_tree      = True,
    core_dist_n_jobs       = -1,
)
hdb_labels     = clusterer.fit_predict(X_scaled)
n_hdb_clusters = len(set(hdb_labels) - {-1})
n_noise        = (hdb_labels == -1).sum()
dbcv           = float(clusterer.relative_validity_)
print(f'Clusters: {n_hdb_clusters}  |  Noise: {n_noise} ({100*n_noise/len(df):.1f}%)')
print(f'DBCV: {dbcv:.4f}  ({"well-separated" if dbcv>0.5 else "moderate" if dbcv>0 else "poor"})')
df['hdb_cluster'] = hdb_labels

## 6. Visualization — UMAP + PCA Sanity Check

Two 2D visualizations of the same cluster labels:
- **UMAP**: nonlinear, preserves local neighborhood structure
- **PCA**: linear, preserves global variance directions

If clusters look consistent in both → results are robust to visualization method ✓
If clusters look very different → the descriptor space has nonlinear structure that
PCA misses (expected), or UMAP is over-fragmenting (worth investigating).

PCA is near-instant at any scale. UMAP uses a single canonical seed (see future
experiments note for stability validation protocol).

In [ ]:
# UMAP
print('Running UMAP...')
reducer = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
    n_components=2, metric='cosine',
    random_state=RANDOM_STATE, low_memory=True,
)
emb_umap = reducer.fit_transform(X_scaled)
df['umap_x'] = emb_umap[:, 0]
df['umap_y'] = emb_umap[:, 1]
print('UMAP done.')

# PCA to 2D (sanity check only — not used for clustering)
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
emb_pca = pca2.fit_transform(X_scaled)
df['pca_x'] = emb_pca[:, 0]
df['pca_y'] = emb_pca[:, 1]
var_explained = pca2.explained_variance_ratio_.sum()
print(f'PCA 2D variance explained: {var_explained:.1%}')
print('Note: if PCA variance < 70%, the descriptor space has substantial variance')
print('in dimensions 3+, meaning PCA 2D is an incomplete picture.')

## 7. Plots

In [ ]:
# ── Plot 1: Library vs Literature ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (emb_x, emb_y, title) in zip(axes, [
    ('umap_x', 'umap_y', 'UMAP'), ('pca_x', 'pca_y', f'PCA ({var_explained:.0%} variance)')]):
    for src in df['source'].unique():
        mask = df['source'] == src
        ax.scatter(df.loc[mask,emb_x], df.loc[mask,emb_y],
                   s=120 if src=='literature' else 6,
                   marker='*' if src=='literature' else 'o',
                   alpha=0.7, label=f'{src} (n={mask.sum():,})',
                   zorder=3 if src=='literature' else 2,
                   edgecolors='white' if src=='literature' else 'none', linewidths=0.5)
    ax.set_title(f'Library vs Literature — {title}', fontweight='bold')
    ax.set_xlabel(f'{title} 1'); ax.set_ylabel(f'{title} 2')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot1_source.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: K-Medoids — UMAP + PCA sanity check ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
cmap = plt.cm.tab10
for ax, (emb_x, emb_y, title) in zip(axes, [
    ('umap_x','umap_y','UMAP'), ('pca_x','pca_y',f'PCA ({var_explained:.0%})')]):
    for lab in sorted(df['km_cluster'].unique()):
        mask = df['km_cluster'] == lab
        ax.scatter(df.loc[mask,emb_x], df.loc[mask,emb_y],
                   c=[cmap(int(lab)%10)], s=6, alpha=0.4,
                   label=f'K{lab} (n={mask.sum():,})', rasterized=True)
    ax.scatter(df.iloc[medoid_idx][emb_x], df.iloc[medoid_idx][emb_y],
               s=200, marker='*', c='black', zorder=10,
               edgecolors='white', linewidths=0.8, label='Medoids')
    ax.set_title(f'K-Medoids (k={N_KMEDOIDS}) — {title}\nSilhouette: {sil_km:.3f}', fontweight='bold')
    ax.set_xlabel(f'{title} 1'); ax.set_ylabel(f'{title} 2')
    ax.legend(fontsize=6, ncol=2)
plt.suptitle('PCA vs UMAP consistency check — clusters should look similar in both', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot2_kmedoids_pca_umap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: HDBSCAN — UMAP only ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
noise_mask = df['hdb_cluster'] == -1
if noise_mask.sum():
    ax.scatter(df.loc[noise_mask,'umap_x'], df.loc[noise_mask,'umap_y'],
               c='lightgrey', s=4, alpha=0.2, label=f'Noise (n={noise_mask.sum():,})', rasterized=True)
cmap_hdb = plt.cm.tab20
for lab in sorted(set(df['hdb_cluster'].unique()) - {-1}):
    mask = df['hdb_cluster'] == lab
    ax.scatter(df.loc[mask,'umap_x'], df.loc[mask,'umap_y'],
               c=[cmap_hdb(int(lab)%20)], s=8, alpha=0.6,
               label=f'C{lab} (n={mask.sum():,})', rasterized=True)
ax.set_title(f'HDBSCAN Natural Clusters ({n_hdb_clusters})\nDBCV: {dbcv:.3f}', fontweight='bold')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot3_hdbscan.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 4: Activity Overlay ───────────────────────────────────────────────────
# Literature IC50/DC50 is the external validation signal:
# if active compounds cluster in specific property regions, those regions
# define the target physicochemical profile for synthesis.
if 'activity_plot' in df.columns and df['activity_plot'].notna().sum() > 0:
    fig, ax = plt.subplots(figsize=(9, 6))
    has_act = df['activity_plot'].notna()
    ax.scatter(df.loc[~has_act,'umap_x'], df.loc[~has_act,'umap_y'],
               c='lightgrey', s=4, alpha=0.15, rasterized=True)
    vals = df.loc[has_act,'activity_plot']
    norm = mcolors.Normalize(vmin=np.percentile(vals,5), vmax=np.percentile(vals,95))
    sc = ax.scatter(df.loc[has_act,'umap_x'], df.loc[has_act,'umap_y'],
                    c=vals, cmap='RdYlGn_r', norm=norm, s=12, alpha=0.8, rasterized=True)
    plt.colorbar(sc, ax=ax, label=f'log10({ACTIVITY_LABEL})' if ACTIVITY_LOG else ACTIVITY_LABEL)
    ax.set_title(f'External Validation — {ACTIVITY_LABEL}\n(green=potent, red=inactive)', fontweight='bold')
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/plot4_activity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No activity data — skipping Plot 4.')

## 8. Cluster Descriptor Profiles

Each K-Medoids cluster characterized by its median descriptor values relative to the
population median. This makes clusters physicochemically interpretable:
e.g., 'Cluster 3 = high-MW, low-TPSA, high-logP compounds — the lipophilic large ring region'

Use this to decide which property regions to prioritize for synthesis.

In [ ]:
pop_medians = df[DESCRIPTORS].median()
pop_iqr     = df[DESCRIPTORS].quantile(0.75) - df[DESCRIPTORS].quantile(0.25)

# Normalized deviation: (cluster median - population median) / population IQR
# Positive = above average, Negative = below average
profile_rows = {}
for lab in sorted(df['km_cluster'].unique()):
    sub = df[df['km_cluster'] == lab]
    cluster_med = sub[DESCRIPTORS].median()
    deviation   = (cluster_med - pop_medians) / pop_iqr
    profile_rows[f'K{lab}'] = deviation

profile_df = pd.DataFrame(profile_rows).T

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(DESCRIPTORS))
width = 0.8 / len(profile_df)
cmap_prof = plt.cm.tab10
for i, (cluster, row) in enumerate(profile_df.iterrows()):
    ax.bar(x + i*width, row.values, width, label=cluster, color=cmap_prof(i%10), alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x + width*(len(profile_df)-1)/2)
ax.set_xticklabels(DESCRIPTORS, rotation=20, ha='right')
ax.set_ylabel('Deviation from population median (IQR units)')
ax.set_title(
    'Cluster Descriptor Profiles\n'
    'Positive = above population median | Negative = below median\n'
    'Use this to characterize each cluster physicochemically',
    fontweight='bold')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot5_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCluster profiles (deviation from population median in IQR units):')
display(profile_df.round(2))
profile_df.to_csv(f'{OUTPUT_DIR}/cluster_descriptor_profiles.csv')

## 9. Inter-Cluster Distance (Cosine)

Measures how distinct clusters are from each other in descriptor space.
- **Low off-diagonal values** = clusters are genuinely distinct property regions ✓
- **High off-diagonal values** = clusters are too similar; consider reducing k

Computed in cosine distance space (consistent with clustering metric).
Full pairwise not feasible at 5M — sample of DIVERSITY_SAMPLE per cluster used.

In [ ]:
labs = sorted(df['km_cluster'].unique())
n_labs = len(labs)

# Sample per cluster
cluster_X = {}
for lab in labs:
    idx = df[df['km_cluster'] == lab].index.tolist()
    sampled = np.random.choice(idx, size=min(DIVERSITY_SAMPLE, len(idx)), replace=False)
    cluster_X[lab] = X_scaled[sampled]

# Intra and inter cosine distance matrix
dist_matrix = np.zeros((n_labs, n_labs))
for i, li in enumerate(labs):
    for j, lj in enumerate(labs):
        Xi = cluster_X[li][:200]
        Xj = cluster_X[lj][:200]
        dist_matrix[i,j] = cosine_distances(Xi, Xj).mean()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(dist_matrix, cmap='Blues', vmin=0)
plt.colorbar(im, ax=ax, label='Mean Cosine Distance')
tick_labels = [f'K{lab}' for lab in labs]
ax.set_xticks(range(n_labs)); ax.set_yticks(range(n_labs))
ax.set_xticklabels(tick_labels, rotation=45, ha='right')
ax.set_yticklabels(tick_labels)
for i in range(n_labs):
    for j in range(n_labs):
        ax.text(j, i, f'{dist_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title(
    'Cluster Distance Heatmap (Cosine)\n'
    'Diagonal = intra-cluster spread | Off-diagonal = inter-cluster distance\n'
    'Low off-diagonal = clusters too similar (consider reducing k)',
    fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plot6_distance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 7: Medoid Structure Grid ─────────────────────────────────────────────
medoid_mols, medoid_legends = [], []
for idx in medoid_idx:
    row = df.iloc[idx]
    mol = Chem.MolFromSmiles(str(row['smiles']))
    if mol:
        AllChem.Compute2DCoords(mol)
        medoid_mols.append(mol)
        props = ' | '.join([f'{d}={row[d]:.1f}' for d in DESCRIPTORS[:3]])
        act_str = f'\n{ACTIVITY_LABEL}={row["activity"]:.1f}' if 'activity' in df.columns and pd.notna(row.get('activity')) else ''
        medoid_legends.append(f'K{km_labels[idx]} | {row["source"]}\n{row["name"]}{act_str}\n{props}')

img = Draw.MolsToGridImage(
    medoid_mols, molsPerRow=4, subImgSize=(320, 270),
    legends=medoid_legends, returnPNG=False)
img.save(f'{OUTPUT_DIR}/plot7_medoid_structures.png')
display(img)
print('Medoid structure grid saved.')

## 10. Outlier Analysis

In [ ]:
noise_df = df[df['hdb_cluster'] == -1][['name','smiles','source'] + DESCRIPTORS +
            (['activity'] if 'activity' in df.columns else [])].copy()
print(f'HDBSCAN noise: {len(noise_df):,} compounds ({100*len(noise_df)/len(df):.1f}%)')
print('These occupy sparse regions of property space — no dense cluster nearby.')
print('Consider: (1) reduce HDBSCAN_MIN_SIZE, (2) flag for manual review, (3) treat as novel targets')
noise_df.to_csv(f'{OUTPUT_DIR}/hdbscan_noise_compounds.csv', index=False)
if len(noise_df) > 0:
    display(noise_df.head(10))

## 11. Synthesis Priority Output

In [ ]:
rows = []
for idx in medoid_idx:
    lab  = int(km_labels[idx])
    mask = df['km_cluster'] == lab
    row  = df.iloc[idx]
    n_lib = (df.loc[mask,'source'] == 'library').sum()
    n_lit = (df.loc[mask,'source'] == 'literature').sum() if LITERATURE_CSV else 0
    coverage = 'overlap' if n_lit>0 and n_lib>0 else 'literature_only' if n_lit>0 else 'library_only'

    other_idx = [i for i, m in enumerate(medoid_idx) if m != idx]
    mean_inter = float(cosine_distances(X_scaled[[idx]], X_scaled[other_idx]).mean()) if other_idx else np.nan

    act_med = float(df.loc[mask,'activity'].median()) if 'activity' in df.columns else None
    desc_profile = {f'median_{d}': round(float(df.loc[mask,d].median()),2) for d in DESCRIPTORS}

    rows.append({
        'cluster':         lab,
        'medoid_name':     row['name'],
        'medoid_smiles':   row['smiles'],
        'medoid_source':   row['source'],
        'n_total':         int(mask.sum()),
        'n_library':       int(n_lib),
        'n_literature':    int(n_lit),
        'coverage':        coverage,
        'mean_inter_dist': round(mean_inter, 3),
        'median_activity': round(act_med, 2) if act_med and not np.isnan(act_med) else None,
        **desc_profile,
    })

synth_df = pd.DataFrame(rows)
coverage_order = {'overlap': 0, 'library_only': 1, 'literature_only': 2}
synth_df['_rank'] = synth_df['coverage'].map(coverage_order)
synth_df = synth_df.sort_values(['_rank','mean_inter_dist'], ascending=[True,False]).drop(columns='_rank')
synth_df.to_csv(f'{OUTPUT_DIR}/synthesis_candidates_desc.csv', index=False)

print('Synthesis priority ranking:')
display(synth_df[['cluster','medoid_name','coverage','n_total','mean_inter_dist','median_activity']].to_string(index=False))

In [ ]:
out_cols = ['name','smiles','source','km_cluster','hdb_cluster','umap_x','umap_y','pca_x','pca_y'] + DESCRIPTORS
if 'activity' in df.columns:
    out_cols.append('activity')
df[out_cols].to_csv(f'{OUTPUT_DIR}/cluster_assignments_desc.csv', index=False)
print(f'Cluster assignments saved: {OUTPUT_DIR}/cluster_assignments_desc.csv')

## 12. QC Summary

In [ ]:
print('=' * 60)
print('QC SUMMARY — Descriptor Framework')
print('=' * 60)
print(f'Representation        : {len(DESCRIPTORS)} bRo5 physicochemical descriptors')
print(f'Descriptors           : {DESCRIPTORS}')
print(f'Compounds analyzed    : {len(df):,}')
print(f'Deduplication removed : {n_exact:,} exact + {n_neardup:,} near-duplicates')
print()
print(f'K-Medoids (k={N_KMEDOIDS}, cosine)')
print(f'  Silhouette           : {sil_km:.4f}  ({"strong" if sil_km>0.5 else "reasonable" if sil_km>0.25 else "weak"})')
print()
print(f'HDBSCAN')
print(f'  Clusters             : {n_hdb_clusters}')
print(f'  Noise                : {n_noise:,} ({100*n_noise/len(df):.1f}%)')
print(f'  DBCV                 : {dbcv:.4f}  ({"well-separated" if dbcv>0.5 else "moderate" if dbcv>0 else "poor"})')
print()
print(f'UMAP                  : cosine, n_neighbors={UMAP_N_NEIGHBORS}, seed={RANDOM_STATE}')
print(f'PCA 2D variance       : {var_explained:.1%} (sanity check only)')
print(f'Outputs               : {OUTPUT_DIR}/')
print('=' * 60)